In [1]:
import os
import shutil
import pandas as pd
from collections import defaultdict

In [2]:
base_dir = "../data/processed_tensor_4s_change"     # chứa A, F, C (ảnh/segment)
output_dir = "../data/dataset_split_4s_change"     # lưu train/val/test
csv_dir = "../data/csv_split_4s_change"            # lưu file CSV mapping

os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

labels = ["A", "F", "C"]

In [4]:
from collections import Counter
segment_counts = Counter()

for label in labels:
    folder = os.path.join(base_dir, label)
    num_segments = len([f for f in os.listdir(folder) if f.endswith(".npy")])
    segment_counts[label] = num_segments

print("Số lượng segment trước khi chia:")
for label, count in segment_counts.items():
    print(f"  {label}: {count} segments")
print(f"  Tổng cộng: {sum(segment_counts.values())} segments")

Số lượng segment trước khi chia:
  A: 4770 segments
  F: 2832 segments
  C: 5309 segments
  Tổng cộng: 12911 segments


In [5]:
# Đọc participants.tsv
participants = pd.read_csv("../data/raw/participants.tsv", sep="\t")
subject_labels = dict(zip(participants["participant_id"], participants["Group"]))

In [6]:
# Đếm số segment mỗi subject
subject_segments = {}
for label in labels:
    folder = os.path.join(base_dir, label)
    for fname in os.listdir(folder):
        subj_id = fname.split("_")[0]  # giả sử tên file bắt đầu bằng sub-xxx
        subject_segments[subj_id] = subject_segments.get(subj_id, 0) + 1

# Chuẩn bị dict chứa split
split_subjects = {"train": defaultdict(list), "val": defaultdict(list), "test": defaultdict(list)}

# Tỷ lệ phân bổ
ratios = {"train": 0.7, "val": 0.15, "test": 0.15}

In [7]:
# Phân bổ subject per label
for label in labels:
    # Lấy tất cả subject của label này
    subs_of_label = [s for s, l in subject_labels.items() if l == label]
    # Sort theo số segment giảm dần
    subs_of_label.sort(key=lambda x: subject_segments[x], reverse=True)

    # Tính tổng segment của label
    total_segments = sum(subject_segments[s] for s in subs_of_label)
    target = {split: total_segments * r for split, r in ratios.items()}
    current = {"train":0, "val":0, "test":0}

    for subj in subs_of_label:
        segs = subject_segments[subj]
        # Chọn split có current < target
        split = min(ratios.keys(), key=lambda s: current[s]/target[s] if target[s]>0 else 1)
        split_subjects[split][label].append(subj)
        current[split] += segs

print("Phân bổ subject xong:")
for split in split_subjects:
    print(f"\n{split.upper()}:")
    for label in labels:
        print(f"{label}: {len(split_subjects[split][label])} subjects, {sum(subject_segments[s] for s in split_subjects[split][label])} segments")

Phân bổ subject xong:

TRAIN:
A: 27 subjects, 3325 segments
F: 17 subjects, 1947 segments
C: 21 subjects, 3701 segments

VAL:
A: 5 subjects, 738 segments
F: 3 subjects, 436 segments
C: 4 subjects, 805 segments

TEST:
A: 4 subjects, 707 segments
F: 3 subjects, 449 segments
C: 4 subjects, 803 segments


In [8]:
# Hàm copy file
def move_subject_images(subjects_dict, split):
    for label, subj_list in subjects_dict.items():
        src_folder = os.path.join(base_dir, label)
        dst_folder = os.path.join(output_dir, split, label)
        os.makedirs(dst_folder, exist_ok=True)

        for fname in os.listdir(src_folder):
            if any(fname.startswith(subj) for subj in subj_list):
                shutil.copy(os.path.join(src_folder, fname), dst_folder)

# Copy dữ liệu
for split in ["train", "val", "test"]:
    move_subject_images(split_subjects[split], split)

In [9]:
# Xuất CSV
import csv
for split in ["train", "val", "test"]:
    rows = []
    for label in labels:
        folder = os.path.join(output_dir, split, label)
        files = os.listdir(folder)
        for f in files:
            filepath = os.path.join(folder, f)
            rows.append([filepath, label])

    csv_file = os.path.join(csv_dir, f"{split}.csv")
    with open(csv_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "label"])
        writer.writerows(rows)
    print(f"Saved {csv_file} with {len(rows)} samples")

Saved ../data/csv_split_4s_change\train.csv with 8973 samples
Saved ../data/csv_split_4s_change\val.csv with 1979 samples
Saved ../data/csv_split_4s_change\test.csv with 1959 samples


In [10]:
# ==== Kiểm tra số subjects trong mỗi split theo label ====
for split in ["train", "val", "test"]:
    df = pd.read_csv(os.path.join(csv_dir, f"{split}.csv"))
    df["subject"] = df["filepath"].str.extract(r"(sub-\d+)")
    print(f"\n===== {split.upper()} set =====")
    for label, group in df.groupby("label"):
        subjects_in_label = sorted(group["subject"].unique())
        print(f"Label {label}: {len(subjects_in_label)} subjects -> {subjects_in_label}")
    print("-"*50)


===== TRAIN set =====
Label A: 27 subjects -> ['sub-001', 'sub-005', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-026', 'sub-027', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036']
Label C: 21 subjects -> ['sub-037', 'sub-038', 'sub-040', 'sub-041', 'sub-042', 'sub-045', 'sub-047', 'sub-048', 'sub-050', 'sub-051', 'sub-052', 'sub-054', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065']
Label F: 17 subjects -> ['sub-066', 'sub-067', 'sub-068', 'sub-070', 'sub-073', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-084', 'sub-085', 'sub-086', 'sub-088']
--------------------------------------------------

===== VAL set =====
Label A: 5 subjects -> ['sub-002', 'sub-013', 'sub-019', 'sub-024', 'sub-025']
Label C: 4 subjects -> ['sub-043', '

In [11]:
# Kiểm tra
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"../data/csv_split_4s/{split}.csv")
    print(f"\n{split.upper()} set:")
    print(df["label"].value_counts())


TRAIN set:
label
C    3701
A    3325
F    1947
Name: count, dtype: int64

VAL set:
label
C    805
A    738
F    436
Name: count, dtype: int64

TEST set:
label
C    803
A    707
F    449
Name: count, dtype: int64
